# GeoNUSAF — two figures: qualitative comparison (Fig. 14) and the leakage dumbbell (Fig. 23)

Builds exactly two files and nothing else:

| file | figure | size | what it is |
|---|---|---|---|
| `fig09_qualitative_comparison.pdf` | Fig. 14 | 156 × 232 mm, `figure*` | 4 scenes × 6 rows; base-tile split above the rule, file-level split below it |
| `fig22_legacy_leakage.pdf` | Fig. 23 | 84 × 46 mm, `figure` | one dumbbell per architecture, base-tile → file-level |

**Run order.** Set the control panel, attach the dataset, turn Internet ON (the checkpoints
come from the Hub), then Run All. A GPU is nice but not needed.

**Three things this notebook refuses to fake.**

1. If a checkpoint will not load, its row draws a labelled placeholder and the reason is
   printed. Nothing is ever substituted for a missing model.
2. Before drawing, it reproduces all three splitters from your training notebooks and
   prints, per scene, whether that scene was *val* or *train* for each of the four runs.
   A panel drawn from a scene the model trained on is not a held-out prediction, and you
   should know which ones those are before the figure goes in the paper.
3. Preprocessing is copied line for line from the four training notebooks: `cv2` BGR read,
   `INTER_AREA` to 512, RGB, `/255`, ImageNet mean/std. Masks are decoded at native
   resolution against the BGR palette, then `INTER_NEAREST` to 512.

## 1 · Control panel

In [ ]:
# =====================================================================================
#  CONTROL PANEL - the only cell you normally edit. Set it, then Run All.
# =====================================================================================

# ---- the four scenes, as file names in the images folder. Leave empty to let the
#      notebook rank the validation tiles by thin-structure content and pick for you
#      (the ranking is printed, so you can overrule it).
#      Four or five entries - the grid follows the length of this list. A fifth column costs
#      panel size: at a fixed 156 mm width the panels go from 35.5 mm square to 28.1 mm, and
#      the figure gets SHORTER (about 187 mm instead of 232), because square panels tie the
#      height to the width. See the note at the end of the notebook.
SCENES = ["", "", "", "", ""]          # e.g. ["image217.png", "image52.png", ...]
SCENE_LABELS = ["scene 1", "scene 2", "scene 3", "scene 4", "scene 5"]

# ---- data
DATA_ROOT = "/kaggle/input/datasets/sugam24/satellite-datasets/satellite_dataset"
#   "auto" walks /kaggle/input looking for images/ + masks/ if the path above is wrong.

# ---- checkpoints.  Rows 3-4 are the base-tile split, rows 5-6 the file-level split.
B0_BASE_REPO = "Pranilllllll/segformer-b0-kaggle-ktm-dataset-random-fold1"
TC_BASE_REPO = "sugam24/geonusaf-tcsegformer-random-fold1"
B0_FILE_REPO = "Pranilllllll/geonusaf-segformer-b0-legacy1-legacy_kfold-fold1"
TC_FILE_REPO = "sugam24/geonusaf-tcsegformer-legacy-c6-tc_full-legacy_kfold-fold1"
CKPT_FILE = "best.pt"
HF_TOKEN = None          # only needed if a repo is private

# ---- which split each run used, so the membership audit is exact.  The two file-level
#      runs did NOT use the same partition: the B0 legacy config is KFold(5), the TC legacy
#      card reports 3-fold, so they are declared separately rather than assumed equal.
BASE_SPLIT = dict(mode="random", fold=1, n_folds=3, seed=42)   # base-tile split, both archs
FILE_SPLIT = {"SegFormer-B0":   dict(fold=1, n_folds=3, seed=42),   # <-- CHECK the B0 card
              "proposed model": dict(fold=1, n_folds=3, seed=42)}   # TC card: 3-fold, fold 1
#      3 folds reproduces the 91.9 % leakage you reported for B0; the notebook's own config
#      default was KFold(5), which reproduces 98.8 % instead. Cell 5 measures it either way
#      and shouts if the measurement disagrees with LEAKAGE below.

# ---- Figure 23 numbers.  Reported values, typed here rather than recomputed.
MIOU = {
    ("SegFormer-B0",   "base"): 0.4657,
    ("SegFormer-B0",   "file"): 0.7806,
    ("proposed model", "base"): 0.5582,
    ("proposed model", "file"): 0.8393,   # pooled, 6 real classes - the same estimator as
}                                         # the B0 file-level 0.7806, NOT the 0.8156
                                          # batch-averaged figure on the same card
LEAKAGE = {"SegFormer-B0": 91.9, "proposed model": 91.9}   # % of val files with a mirror in train
XLIM_23 = (0.40, 0.88)     # 0.85 would put the proposed filled marker on the right spine

# ---- output
OUT = "figs"
SAVE_PNG = True          # 600 dpi previews for checking; submit the PDFs
CACHE_PREDS = True       # keep predictions in figs/_cache so re-runs are instant

print("control panel set")


## 2 · Environment — nothing is installed and the kernel is not restarted

The earlier version of this cell pinned `transformers<5` on the claim that 5.x renames
`decode_head.linear_c`. That claim was wrong: diffing 4.57.6 against 5.0.0, the SegFormer
module is unchanged apart from type-hint syntax, the removal of head pruning and a moved
`_init_weights` — every parameter name is identical, so both builds load these checkpoints.
The pin was causing a reinstall and a kernel restart on every fresh Kaggle session for no
reason, so it is gone.

What replaces it is a check of the thing that actually matters: the six
`SegformerDecodeHead` attributes `TCSegFormer` reaches into. If a future build renames one,
this raises here, and cell 6's 98 % key-coverage test catches it again before any model is
used.

In [ ]:
# =====================================================================================
#  Environment - no installs, no kernel restart.
#
#  What actually matters is not the transformers version number but whether this build
#  names the decode head the way your checkpoints do. TCSegFormer reaches into six
#  attributes of SegformerDecodeHead, and a build that renamed any of them would load
#  garbage. So the check below reads the installed source and confirms the six names.
#  (Verified present in both 4.57.6 and 5.0.0 - the SegFormer module is unchanged between
#  them apart from type-hint syntax and the removal of head pruning.)
#
#  Cell 6 is the second net: it rejects any checkpoint whose keys match under 98 % rather
#  than part-loading it, so a future build that did rename something would fail loudly
#  there instead of drawing a wrong panel.
# =====================================================================================

import inspect
import torch, cv2, numpy as np, transformers
from transformers.models.segformer import modeling_segformer as _ms

NEEDED = ["linear_c", "linear_fuse", "batch_norm", "activation", "dropout", "classifier"]
_src = inspect.getsource(_ms.SegformerDecodeHead)
_missing = [a for a in NEEDED if f"self.{a} = " not in _src]

print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| cv2", cv2.__version__)
if _missing:
    raise RuntimeError(
        "This transformers build renames SegformerDecodeHead attributes "
        + str(_missing) + ", so the checkpoint keys will not match. Install a build that "
        "keeps them (`pip install -q 'transformers==4.57.6'`), then RESTART THE KERNEL "
        "before running anything else - installing over an imported transformers leaves "
        "two versions in one folder and every later import fails.")
print("SegformerDecodeHead exposes all six attributes the TC class needs:", ", ".join(NEEDED))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "(CPU works too - four models over four scenes is about a minute)")


## 3 · House style and palette

Same Times fallback chain, same display palette and the same `pdf.fonttype 42` as the main figures notebook, so these two figures sit beside the other twenty-one without looking imported.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle, FancyArrowPatch
from matplotlib.lines import Line2D
import matplotlib.font_manager as fm

OUTDIR = Path(OUT); OUTDIR.mkdir(exist_ok=True)
CACHE = OUTDIR / "_cache"; CACHE.mkdir(exist_ok=True)
MM = 1 / 25.4

# Liberation Serif and Nimbus Roman are metrically identical to Times New Roman.
SERIF = ["Times New Roman", "Nimbus Roman", "Liberation Serif", "TeX Gyre Termes",
         "FreeSerif", "DejaVu Serif"]
have = {f.name for f in fm.fontManager.ttflist}
resolved = next((f for f in SERIF if f in have), "DejaVu Serif")
print("serif family in use:", resolved)
if resolved == "DejaVu Serif":
    print("  Times not found. On Kaggle:  !apt-get -qq install -y fonts-liberation")
    print("  then restart the kernel. DejaVu Serif is a safe stand-in meanwhile.")

mpl.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 600,
    "savefig.facecolor": "white",
    "savefig.transparent": False,
    "pdf.fonttype": 42,            # embed TrueType; Type 3 fails Elsevier preflight
    "ps.fonttype": 42,
    "font.family": "serif",
    "font.serif": SERIF,
    "mathtext.fontset": "stix",
    "font.size": 7,
    "axes.linewidth": 0.6,
    "legend.frameon": False,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2.4,  "ytick.major.size": 2.4,
    "xtick.labelsize": 7, "ytick.labelsize": 7,
})

# ---- classes.  Model index order is the training order; the legend order is the one
#      every other figure in the paper uses.
TRAIN_ORDER  = ["Residential", "Road", "River", "Forest", "UnusedLand", "Agricultural"]
LEGEND_ORDER = ["Residential", "Agricultural", "UnusedLand", "Forest", "Road", "River"]

DISPLAY = {"Residential": "#C0392B", "Agricultural": "#E8B62C", "UnusedLand": "#C2A477",
           "Forest": "#14532D", "Road": "#4D4D4D", "River": "#1F6FB4",
           "Unannotated": "#FFFFFF"}
CLR = DISPLAY

ACLR = {"SegFormer-B0": "#2C7FB8", "proposed model": "#B2182B"}

def hex2rgb(h):
    h = h.lstrip("#")
    return np.array([int(h[i:i + 2], 16) for i in (0, 2, 4)], np.uint8)

# index -> display RGB, with 255 (unannotated / unlabelled) rendered as the palette's white
PAINT = np.zeros((256, 3), np.uint8)
PAINT[:] = hex2rgb(CLR["Unannotated"])
for i, c in enumerate(TRAIN_ORDER):
    PAINT[i] = hex2rgb(CLR[c])

def paint(idx):
    """class-index map -> RGB image in the figure palette."""
    return PAINT[np.clip(idx, 0, 255).astype(np.uint8)] if idx is not None else None

# ---- mask decoding.  Masks are cv2-written, so the colours in the file are BGR tuples.
BGR_TO_CLASS = {(128, 0, 0): 0, (0, 128, 0): 1, (0, 0, 128): 2,
                (0, 128, 128): 3, (128, 128, 0): 4, (128, 0, 128): 5}
LUT = np.full(1 << 24, 255, np.uint8)
for bgr, i in BGR_TO_CLASS.items():
    LUT[(bgr[0] << 16) | (bgr[1] << 8) | bgr[2]] = i

def pack_bgr(a):
    a = a.astype(np.uint32)
    return (a[..., 0] << 16) | (a[..., 1] << 8) | a[..., 2]

IMNET_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
IMNET_STD  = np.array([0.229, 0.224, 0.225], np.float32)
IMG_SIZE = 512
print("palette: display · classes:", ", ".join(TRAIN_ORDER))


## 4 · Dataset

In [ ]:
import os, re, cv2
from pathlib import Path

PAT = re.compile(r"^image(\d+)(hf|vf)?\.png$", re.I)

def _find_root(start="/kaggle/input", depth=5):
    start = Path(start)
    if not start.exists():
        return None
    stack = [(start, 0)]
    while stack:
        d, k = stack.pop()
        try:
            subs = [p for p in d.iterdir() if p.is_dir()]
        except Exception:
            continue
        names = {p.name.lower() for p in subs}
        if "images" in names and "masks" in names:
            return d
        if k < depth:
            stack += [(p, k + 1) for p in subs]
    return None

root = Path(DATA_ROOT)
if DATA_ROOT == "auto" or not (root / "images").exists():
    found = _find_root()
    assert found is not None, ("dataset not found. Set DATA_ROOT in the control panel to the "
                              "folder that contains images/ and masks/.")
    root = found
IMG_DIR, MASK_DIR = root / "images", root / "masks"
print("data root:", root)

# records in exactly the order the training notebooks built them: sorted(os.listdir)
RECORDS = []                       # (base_id, variant, image_name, mask_name)
for f in sorted(os.listdir(IMG_DIR)):
    m = PAT.match(f)
    if not m:
        continue
    RECORDS.append((int(m.group(1)), (m.group(2) or "orig"), f, "mask" + f[len("image"):]))
BASE_IDS = sorted({r[0] for r in RECORDS})
GROUPS = {}
for r in RECORDS:
    GROUPS.setdefault(r[0], []).append(r)
print(f"{len(RECORDS)} files | {len(BASE_IDS)} base tiles "
      f"(expected 1224 / 408)")

NAME_OF = {r[2]: r for r in RECORDS}

def resolve_scene(s):
    """Accept 'image217.png', 'image217', or a full path. Returns the record, or None."""
    if not s:
        return None
    s = Path(str(s)).name
    if s in NAME_OF:
        return NAME_OF[s]
    if (s + ".png") in NAME_OF:
        return NAME_OF[s + ".png"]
    return None

def read_image(rec):
    a = cv2.imread(str(IMG_DIR / rec[2]), cv2.IMREAD_COLOR)          # BGR, native size
    return None if a is None else a

def read_gt(rec):
    """Decode at native resolution, then nearest-resize to 512 - exactly as training does."""
    m = cv2.imread(str(MASK_DIR / rec[3]), cv2.IMREAD_COLOR)
    if m is None:
        return None
    lab = LUT[pack_bgr(m)]
    return cv2.resize(lab, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

def rgb512(rec):
    a = read_image(rec)
    if a is None:
        return None
    a = cv2.resize(a, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return cv2.cvtColor(a, cv2.COLOR_BGR2RGB)


## 5 · Splits, the four scenes, and the membership audit

The two clean runs do **not** share a validation set: `3__segformer-b0.ipynb` shuffles with
`random.Random(42)` and `5__tc-segformer.ipynb` with `np.random.RandomState(42)`, so their
random folds differ. The legacy runs split over the 1224 **files**, which is the leakage
itself, and each declares its own fold count because they need not match.

The leakage percentage is **measured** here rather than copied from the model cards, and the
cell says so loudly if the measurement disagrees with `LEAKAGE`. `KFold(3)` fold 1 gives 375
of 408 val files with a mirror in training — 91.9 %, exactly what the TC legacy card
reports, which is also a check that the split is reproduced correctly. `KFold(5)` would give
98.8 % instead.

In [ ]:
import random, numpy as np
from sklearn.model_selection import KFold

# ---------------------------------------------------------------- the three splitters,
# copied from the notebooks that produced the checkpoints. They are NOT interchangeable:
# the two clean notebooks shuffle with different RNGs, so their random folds differ.

def random_split_b0(base_ids, fold, n_folds, seed):          # 3__segformer-b0.ipynb
    ids = list(base_ids)
    random.Random(seed).shuffle(ids)
    e = [round(i * len(ids) / n_folds) for i in range(n_folds + 1)]
    return sorted(ids[:e[fold]] + ids[e[fold + 1]:]), sorted(ids[e[fold]:e[fold + 1]])

def random_split_tc(base_ids, fold, n_folds, seed):          # 5__tc-segformer.ipynb
    ids = np.array(sorted(base_ids))
    np.random.RandomState(seed).shuffle(ids)
    chunks = np.array_split(ids, n_folds)
    val = set(int(v) for v in chunks[fold])
    return sorted(set(int(v) for v in ids) - val), sorted(val)

def block_split(base_ids, fold, n_folds, buffer=2):
    n = len(base_ids)
    e = [round(i * n / n_folds) for i in range(n_folds + 1)]
    lo, hi = e[fold], e[fold + 1]
    return base_ids[:max(0, lo - buffer)] + base_ids[min(n, hi + buffer):], base_ids[lo:hi]

def legacy_kfold(n_items, fold, n_folds, seed):              # legacy notebooks, over FILES
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for i, (tr, va) in enumerate(kf.split(np.arange(n_items))):
        if i == fold:
            return list(tr), list(va)
    raise IndexError(fold)

_mode, _fold, _nf, _seed = (BASE_SPLIT["mode"], BASE_SPLIT["fold"],
                            BASE_SPLIT["n_folds"], BASE_SPLIT["seed"])
if _mode == "block":
    b0_tr, b0_va = block_split(BASE_IDS, _fold, _nf)
    tc_tr, tc_va = b0_tr, b0_va
else:
    b0_tr, b0_va = random_split_b0(BASE_IDS, _fold, _nf, _seed)
    tc_tr, tc_va = random_split_tc(BASE_IDS, _fold, _nf, _seed)

# Each file-level run gets its own partition: the B0 legacy config is KFold(5), the TC
# legacy card reports 3-fold. Leakage is MEASURED here, not taken on trust, so you can
# check it against the percentage printed on each model card.
LG = {}
for _arch, _spec in FILE_SPLIT.items():
    _tr_i, _va_i = legacy_kfold(len(RECORDS), _spec["fold"], _spec["n_folds"], _spec["seed"])
    _tr = {RECORDS[i][2] for i in _tr_i}
    _va = {RECORDS[i][2] for i in _va_i}
    _leak = [r for r in RECORDS if r[2] in _va
             and any(s[2] in _tr for s in GROUPS[r[0]] if s[2] != r[2])]
    _pct = 100 * len(_leak) / max(len(_va), 1)
    LG[_arch] = dict(train=_tr, val=_va, leak_pct=_pct)
    _card = LEAKAGE.get(_arch)
    _flag = "" if _card is None or abs(_pct - _card) < 0.25 else \
            f"   <-- control panel says {_card:.1f}%, CHECK WHICH IS RIGHT"
    print(f"file-level split, {_arch}: KFold({_spec['n_folds']}) fold {_spec['fold']} -> "
          f"{len(_va)} val files | mirror in train for {len(_leak)} of them "
          f"({_pct:.1f}%){_flag}")

# ---------------------------------------------------------------- choose the four scenes
def thin_share(rec, maxside=768):
    """road + river share of the tile, measured on a downscaled mask (fast)."""
    m = cv2.imread(str(MASK_DIR / rec[3]), cv2.IMREAD_COLOR)
    if m is None:
        return 0.0
    if max(m.shape[:2]) > maxside:
        m = cv2.resize(m, (maxside, maxside), interpolation=cv2.INTER_NEAREST)
    lab = LUT[pack_bgr(m)]
    return float(((lab == 1) | (lab == 2)).mean())

picked = [resolve_scene(s) for s in SCENES]
bad = [s for s, r in zip(SCENES, picked) if s and r is None]
assert not bad, f"these scene names are not in the images folder: {bad}"

if any(r is None for r in picked):
    pool = sorted(set(b0_va) & set(tc_va))          # held out by BOTH base-tile models
    if not pool:
        pool = sorted(set(b0_va) | set(tc_va))
    cand = [GROUPS[t][0] for t in pool]
    cand = [r for r in cand if r[1] == "orig"]
    scored = sorted(((thin_share(r), r) for r in cand), reverse=True)[:12]
    print("\nno scenes set - ranking held-out tiles by road+river share:")
    for v, r in scored:
        print(f"   {r[2]:<16} {100*v:5.2f}%")
    fill = [r for _, r in scored]
    for i in range(len(picked)):
        if picked[i] is None and fill:
            picked[i] = fill.pop(0)
    print("using:", [r[2] for r in picked])

assert all(r is not None for r in picked), "could not choose enough scenes"
assert len({r[2] for r in picked}) == len(picked), "the scenes must be distinct"
assert len(SCENE_LABELS) >= len(picked), "SCENE_LABELS needs one entry per scene"

# ---------------------------------------------------------------- membership audit
_b0va, _tcva = set(b0_va), set(tc_va)
print("\nscene              base-tile split        file-level split")
print("                   B0       proposed      B0       proposed   mirror of it in train")
for r in picked:
    sibs = [s[2] for s in GROUPS[r[0]] if s[2] != r[2]]
    mir = [a for a in FILE_SPLIT if any(x in LG[a]["train"] for x in sibs)]
    print(f"  {r[2]:<16} {'val' if r[0] in _b0va else 'TRAIN':<8} "
          f"{'val' if r[0] in _tcva else 'TRAIN':<13} "
          f"{'val' if r[2] in LG['SegFormer-B0']['val'] else 'TRAIN':<8} "
          f"{'val' if r[2] in LG['proposed model']['val'] else 'TRAIN':<10} "
          f"{', '.join(mir) if mir else 'neither'}")
print("\nTRAIN in any column means that model saw that scene while training, so its panel is "
      "not a held-out prediction.\nThe last column is the leakage itself: expected under the "
      "file-level split, and what rows 5-6 exist to show.")


## 6 · Checkpoints

Four different save formats are handled by detecting the tensor dict by content rather than
by key name (`model_state` in the two B0 runs and the legacy TC recipe, `model` in the clean
TC run). The TC class here names the residual head `refine`, which is what the training
class stored; a checkpoint that stores it as `classifier` is remapped. A model whose keys
match under 98 % is rejected rather than partly loaded.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from huggingface_hub import hf_hub_download
from transformers import SegformerConfig, SegformerForSemanticSegmentation

DEFAULT_BACKBONE = "nvidia/segformer-b0-finetuned-ade-512-512"

def _tensor_state(ck):
    """The four runs save under three different key names, so detect by content."""
    for k in ("model_state", "model", "state_dict"):
        v = ck.get(k)
        if isinstance(v, dict) and v and all(torch.is_tensor(t) for t in v.values()):
            return v
    for v in ck.values():
        if isinstance(v, dict) and v and all(torch.is_tensor(t) for t in v.values()):
            return v
    raise KeyError("no tensor state dict in this checkpoint")

def _coverage(model, sd):
    have = model.state_dict()
    ok = sum(1 for k, v in sd.items() if k in have and tuple(have[k].shape) == tuple(v.shape))
    return ok / max(len(sd), 1)

# ------------------------------------------------------------------ SegFormer-B0
def build_b0(ck):
    mc = ck.get("model_config")
    if isinstance(mc, dict):
        conf = SegformerConfig.from_dict(mc)
    else:
        C = ck.get("cfg") or {}
        conf = SegformerConfig.from_pretrained(C.get("HF_MODEL_ID", DEFAULT_BACKBONE))
        conf.num_labels = int(C.get("N_CLASSES") or C.get("NUM_CLASSES") or 6)
    return SegformerForSemanticSegmentation(conf)

# ------------------------------------------------------------------ TC-SegFormer
class _Detail(nn.Module):
    def __init__(s, chs):
        super().__init__()
        c1, c2, c3 = chs
        b = lambda i, o, st: nn.Sequential(nn.Conv2d(i, o, 3, st, 1, bias=False),
                                           nn.GroupNorm(min(8, o), o), nn.ReLU(inplace=True))
        s.net = nn.Sequential(b(3, c1, 2), b(c1, c2, 1), b(c2, c3, 1)); s.out_ch = c3
    def forward(s, x): return s.net(x)

class TCSegFormer(nn.Module):
    """Identical to the class in 5__tc-segformer.ipynb and tc-segformer-legacy-recipe.ipynb.
    The residual head is named `refine`, which is what both checkpoints store."""
    def __init__(s, C):
        super().__init__()
        conf = SegformerConfig.from_pretrained(C["backbone"])
        conf.num_labels = C["n_classes"]
        base = SegformerForSemanticSegmentation(conf)      # weights come from the checkpoint
        s.encoder, s.head = base.segformer, base.decode_head
        s.use_detail = C["use_detail"]
        if s.use_detail:
            s.proj = nn.Conv2d(conf.decoder_hidden_size, C["fuse"], 1)
            s.detail = _Detail(C["detail"])
            s.gate = nn.Parameter(torch.tensor(float(C["gate"])))
            s.fuse = nn.Sequential(
                nn.Conv2d(C["fuse"] + s.detail.out_ch, C["fuse"], 3, 1, 1, bias=False),
                nn.GroupNorm(8, C["fuse"]), nn.ReLU(inplace=True))
            s.refine = nn.Conv2d(C["fuse"], C["n_classes"], 1)

    def forward(s, x):
        H, W = x.shape[-2:]
        hs = s.encoder(x, output_hidden_states=True, return_dict=True).hidden_states
        outs = []
        for h_, mlp in zip(hs, s.head.linear_c):
            t = mlp(h_).permute(0, 2, 1).reshape(h_.shape[0], -1, h_.shape[2], h_.shape[3])
            outs.append(F.interpolate(t, size=hs[0].shape[2:], mode="bilinear",
                                      align_corners=False))
        fused = s.head.dropout(s.head.activation(s.head.batch_norm(
            s.head.linear_fuse(torch.cat(outs[::-1], 1)))))
        logits = F.interpolate(s.head.classifier(fused), size=(H, W),
                               mode="bilinear", align_corners=False)
        if not s.use_detail:
            return logits
        f = F.interpolate(s.proj(fused), size=(H // 2, W // 2), mode="bilinear",
                          align_corners=False)
        f = torch.cat([f, s.gate * s.detail(x)], 1)
        return logits + F.interpolate(s.refine(s.fuse(f)), size=(H, W),
                                      mode="bilinear", align_corners=False)

def _tc_cfg(C):
    C = C or {}
    return dict(backbone=C.get("BACKBONE") or C.get("HF_MODEL_ID") or DEFAULT_BACKBONE,
                n_classes=int(C.get("NUM_CLASSES") or C.get("N_CLASSES") or 6),
                fuse=int(C.get("FUSE_CH", 64)),
                detail=tuple(C.get("DETAIL_CH", (8, 16, 32))),
                use_detail=bool(C.get("USE_DETAIL", True)),
                gate=float(C.get("DETAIL_GATE_INIT", 0.1)))

def build_tc(ck):
    return TCSegFormer(_tc_cfg(ck.get("cfg")))

# ------------------------------------------------------------------ one entry point
def load_model(repo, kind, slot):
    """Returns (model, kind, n_classes, note) or (None, kind, 0, why-it-failed)."""
    if not repo:
        return None, kind, 0, "no checkpoint given for " + slot
    try:
        p = hf_hub_download(repo, CKPT_FILE, token=HF_TOKEN)
        ck = torch.load(p, map_location="cpu", weights_only=False)
        sd = _tensor_state(ck)
        net = build_b0(ck) if kind == "b0" else build_tc(ck)
        # the standalone loader in 5__tc-segformer.ipynb names the residual head
        # `classifier`; the training class (and therefore the checkpoint) names it `refine`
        if kind == "tc" and any(k.startswith("classifier.") for k in sd):
            sd = {("refine." + k[len("classifier."):] if k.startswith("classifier.") else k): v
                  for k, v in sd.items()}
        cov = _coverage(net, sd)
        if cov < 0.98:
            return None, kind, 0, f"{repo}: only {100*cov:.0f}% of checkpoint keys match"
        miss, unexp = net.load_state_dict(sd, strict=False)
        net.to(DEVICE).eval()
        n_cls = (int(net.config.num_labels) if kind == "b0"
                 else int(_tc_cfg(ck.get("cfg"))["n_classes"]))
        rep = (ck.get("metrics_full") or ck.get("metrics") or {})
        got = next((rep[k] for k in ("mIoU", "mIoU_core6", "miou") if k in rep), None)
        note = (f"{repo} | {n_cls} classes | keys {100*cov:.1f}% "
                f"({len(miss)} missing, {len(unexp)} unexpected)"
                + (f" | stored mIoU {got:.4f}" if isinstance(got, float) else ""))
        return net, kind, n_cls, note
    except Exception as e:
        return None, kind, 0, f"{repo}: {type(e).__name__}: {str(e).splitlines()[0][:180]}"

ROWS = [("SegFormer-B0",   "base", B0_BASE_REPO, "b0"),
        ("proposed model", "base", TC_BASE_REPO, "tc"),
        ("SegFormer-B0",   "file", B0_FILE_REPO, "b0"),
        ("proposed model", "file", TC_FILE_REPO, "tc")]

MODELS, NOTES = {}, {}
for arch, split, repo, kind in ROWS:
    slot = f"{arch} / {'base-tile' if split=='base' else 'file-level'} split"
    net, kind, n_cls, note = load_model(repo, kind, slot)
    MODELS[(arch, split)] = (net, kind, n_cls)
    NOTES[(arch, split)] = note
    print(("  OK   " if net is not None else "  MISS ") + slot)
    print("         " + note)

print("\nA row with no model draws a labelled placeholder. Nothing is ever substituted.")


## 7 · Predictions

In [ ]:
import numpy as np, torch, torch.nn.functional as F, cv2, hashlib

@torch.no_grad()
def predict(entry, rec, tag=""):
    """Preprocessing copied from all four training notebooks: cv2 BGR read, INTER_AREA to
    512, RGB, /255, ImageNet mean/std. A 7-class head's 'unlabelled' becomes 255."""
    net, kind, n_cls = entry
    if net is None:
        return None
    key = hashlib.md5(f"{tag}|{rec[2]}".encode()).hexdigest()[:10]
    f = CACHE / f"pred_{key}.npy"
    if CACHE_PREDS and f.exists():
        return np.load(f)
    im = cv2.imread(str(IMG_DIR / rec[2]), cv2.IMREAD_COLOR)
    if im is None:
        return None
    im = cv2.resize(im, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    x = ((cv2.cvtColor(im, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0)
         - IMNET_MEAN) / IMNET_STD
    x = torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1)))[None].to(DEVICE)
    with torch.autocast(device_type=("cuda" if DEVICE == "cuda" else "cpu"),
                        enabled=(DEVICE == "cuda")):
        lo = net(pixel_values=x).logits if kind == "b0" else net(x)
    if lo.shape[-2:] != (IMG_SIZE, IMG_SIZE):
        lo = F.interpolate(lo.float(), size=(IMG_SIZE, IMG_SIZE), mode="bilinear",
                           align_corners=False)
    p = lo.float().argmax(1)[0].cpu().numpy().astype(np.int16)
    p[p >= 6] = 255                       # 7-class legacy scheme: class 6 is unlabelled
    if CACHE_PREDS:
        np.save(f, p)
    return p

REPO_OF = {(a, s): rp for a, s, rp, _ in ROWS}
PANELS = {}                                # (scene_name, row_key) -> RGB array or None
for rec in picked:
    PANELS[(rec[2], "image")] = rgb512(rec)
    PANELS[(rec[2], "gt")] = paint(read_gt(rec))
    for arch, split, _, _ in ROWS:
        idx = predict(MODELS[(arch, split)], rec, tag=f"{arch}|{split}|{REPO_OF[(arch, split)]}")
        PANELS[(rec[2], (arch, split))] = paint(idx) if idx is not None else None
missing = sorted({k[1] for k, v in PANELS.items() if v is None})
print("panels built:", sum(v is not None for v in PANELS.values()), "of", len(PANELS))
if missing:
    print("rows with no output:", missing)


## 8 · Figure 14 — `fig09_qualitative_comparison.pdf`

The grid follows the length of `SCENES`. Panels are square, so the 156 mm width fixes
everything else:

| columns | panel | page | printed panel at 0.94 | dpi for a 512 px tile |
|---|---|---|---|---|
| 4 | 35.5 mm | 156 × 232 mm | 33.4 mm | ~390 |
| 5 | 28.1 mm | 156 × 186 mm | 26.4 mm | ~490 |

A fifth column cannot be had at the same height: six rows of square panels tie the height to
the panel size, so the extra column makes every panel 21 % smaller and the figure *shorter*,
not taller. Resolution is not the casualty — 490 dpi is comfortably above the 300 dpi floor —
but apparent detail is, and thin structure is what this figure exists to show. Going back is
deleting one entry from `SCENES`.

9 mm is reserved on the left for the rotated row labels and 6 mm at the top for the legend.
All lettering is 9 pt, landing at 8.5 pt on the page. At five columns the row-2 label
switches to two lines automatically, because "reference annotation" no longer fits along a
28 mm panel.

In [ ]:
# =====================================================================================
#  Figure 14 - fig09_qualitative_comparison.pdf
#  156 x 232 mm portrait, 4 scenes x 6 rows. Prints at ~147 x 218 mm (scale 0.94), so all
#  lettering is set at 9 pt and lands at 8.5 pt on the page.
# =====================================================================================

ROW_KEYS = ["image", "gt",
            ("SegFormer-B0", "base"), ("proposed model", "base"),
            ("SegFormer-B0", "file"), ("proposed model", "file")]
ROW_LABELS = ["image",
              "reference annotation",   # row 2 is shortened to two lines when panels shrink
              "SegFormer-B0\nbase-tile split",
              "proposed model\nbase-tile split",
              "SegFormer-B0\nfile-level split",
              "proposed model\nfile-level split"]

FS = 9.0                      # every piece of lettering in this figure
NCOL = len(picked)            # 4 or 5; the geometry follows it
W14 = 156 * MM                # page-limited: the text block is about 147 mm wide
L, R, BM, LEG, GUT, HDR = 9 * MM, 1.5 * MM, 1.5 * MM, 6 * MM, 1.2 * MM, 4.15 * MM

# Panels are square, so the width fixes everything: more columns means smaller panels and,
# because the six rows then need less room, a SHORTER figure. 4 columns -> 35.5 mm panels
# and a 232 mm page; 5 columns -> 28.1 mm panels and about 187 mm. A 512 px tile still
# prints at ~490 dpi in the 5-column layout, so resolution is not the constraint; apparent
# detail is.
SW = (W14 - L - R - (NCOL - 1) * GUT) / NCOL
SH = SW
H14 = BM + 6 * SH + 5 * GUT + HDR + LEG
GRID_R = (L + NCOL * SW + (NCOL - 1) * GUT) / W14      # right edge of the panel block
print(f"grid: {NCOL} x 6 | panel {SW/MM:.1f} mm square | page {W14/MM:.0f} x {H14/MM:.0f} mm")

def _panel_box(r, c):
    x0 = L + c * (SW + GUT)
    y1 = H14 - LEG - HDR - r * (SH + GUT)
    return [x0 / W14, (y1 - SH) / H14, SW / W14, SH / H14]

def build_fig14():
    fig = plt.figure(figsize=(W14, H14))

    # light tint behind the two file-level rows, so they read as a different protocol
    ytop = (H14 - LEG - HDR - 4 * (SH + GUT) + 0.6 * MM) / H14
    ybot = (H14 - LEG - HDR - 5 * (SH + GUT) - SH - 0.6 * MM) / H14
    fig.add_artist(Rectangle((0.004, ybot), 0.992, ytop - ybot, transform=fig.transFigure,
                             fc="#F1F4F7", ec="none", zorder=0))

    for r, key in enumerate(ROW_KEYS):
        for c, rec in enumerate(picked):
            ax = fig.add_axes(_panel_box(r, c))
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_linewidth(0.5); sp.set_color("0.55")
            arr = PANELS.get((rec[2], key))
            if arr is None:
                ax.add_patch(Rectangle((0, 0), 1, 1, transform=ax.transAxes,
                                       fc="#F4F4F4", ec="#B2182B", lw=0.8))
                txt = ("checkpoint\nnot yet trained"
                       if (isinstance(key, tuple) and not REPO_OF.get(key))
                       else "output\nnot available")
                ax.text(0.5, 0.5, txt, transform=ax.transAxes, ha="center", va="center",
                        fontsize=FS - 1.5, color="#B2182B")
            else:
                ax.imshow(arr, interpolation="nearest")
            if r == 0:
                fig.text(_panel_box(0, c)[0] + _panel_box(0, c)[2] / 2,
                         (H14 - LEG - HDR / 2) / H14, SCENE_LABELS[c],
                         ha="center", va="center", fontsize=FS)
        box = _panel_box(r, 0)
        lbl = ROW_LABELS[r]
        if r == 1 and SH < 32 * MM:            # would not fit on one line at this panel size
            lbl = "reference\nannotation"
        fig.text((L / 2) / W14, box[1] + box[3] / 2, lbl, rotation=90,
                 ha="center", va="center", multialignment="center", fontsize=FS)

    # the rule that marks where the protocol changes
    yr = (H14 - LEG - HDR - 4 * (SH + GUT) + GUT / 2) / H14
    fig.add_artist(Line2D([0.004, GRID_R], [yr, yr],
                          transform=fig.transFigure, color="0.15", lw=0.8))

    handles = [Patch(fc=CLR[c], ec="0.35", lw=0.4) for c in LEGEND_ORDER] + \
              [Patch(fc=CLR["Unannotated"], ec="0.35", lw=0.4)]
    labels = LEGEND_ORDER + ["unannotated"]
    fig.legend(handles=handles, labels=labels, loc="upper center",
               bbox_to_anchor=(0.5, 1.0 - 0.2 * MM / H14), ncol=7, fontsize=FS,
               frameon=False, handlelength=1.0, handleheight=0.9, handletextpad=0.35,
               columnspacing=0.9, borderaxespad=0.0, borderpad=0.0)

    fig.savefig(OUTDIR / "fig09_qualitative_comparison.pdf")
    if SAVE_PNG:
        fig.savefig(OUTDIR / "fig09_qualitative_comparison.png")
    print("fig09_qualitative_comparison.pdf  %d x %d mm  figure*  (Fig. 14, end of S6.2)"
          % (round(W14 / MM), round(H14 / MM)))
    return fig

_f14 = build_fig14()
plt.show()


## 9 · Figure 23 — `fig22_legacy_leakage.pdf`

Two things deliberately absent: no connector, bracket or bar between the two filled markers,
because that is the cross-architecture comparison the paper refuses on the grounds that the
two runs leaked by different amounts; and no error bars, because this is one fold and one
seed.

In [ ]:
# =====================================================================================
#  Figure 23 - fig22_legacy_leakage.pdf
#  84 x 46 mm, single column, one message: what the file-level split adds to each model.
#  Deliberately absent: any connector between the two filled markers (the two runs leaked
#  by different amounts, so they are not comparable), and any error bar (one fold, one seed).
# =====================================================================================

ARCHS = ["SegFormer-B0", "proposed model"]        # top row first
W23, H23 = 84 * MM, 46 * MM

def _delta(a, b):
    """subtract in decimal and round half-up, so 0.7987-0.5582 prints as 0.241 not 0.240"""
    from decimal import Decimal, ROUND_HALF_UP
    d = Decimal(repr(b)) - Decimal(repr(a))
    return str(d.quantize(Decimal("0.001"), rounding=ROUND_HALF_UP))

def build_fig23():
    fig = plt.figure(figsize=(W23, H23))
    ax = fig.add_axes([19.0 / 84, 9.0 / 46, (84 - 19.0 - 15.0) / 84, (46 - 9.0 - 5.5) / 46])

    for i, arch in enumerate(ARCHS):
        y = 1 - i                                   # SegFormer-B0 on top
        a, b = MIOU[(arch, "base")], MIOU[(arch, "file")]
        c = ACLR[arch]
        ax.add_patch(FancyArrowPatch((a + 0.012, y), (b - 0.018, y),
                                     arrowstyle="-|>", mutation_scale=6.5, lw=0.9,
                                     color=c, shrinkA=0, shrinkB=0, zorder=2))
        ax.plot([a], [y], "o", mfc="white", mec=c, mew=1.0, ms=4.6, zorder=3, clip_on=False)
        ax.plot([b], [y], "o", mfc=c, mec=c, ms=4.6, zorder=3, clip_on=False)
        ax.text((a + b) / 2, y + 0.16, "+" + _delta(a, b), ha="center", va="bottom",
                fontsize=7, color=c)
        ax.text(1.03, y, "leakage %.1f%%" % LEAKAGE[arch], transform=ax.get_yaxis_transform(),
                ha="left", va="center", fontsize=6.3, color="0.45", clip_on=False)

    ax.set_xlim(*XLIM_23)
    ax.set_ylim(-0.62, 1.62)
    ax.set_yticks([1, 0]); ax.set_yticklabels(ARCHS, fontsize=7)
    ax.set_xlabel("mean intersection over union", fontsize=7.5)
    ax.set_xticks(np.arange(0.40, 0.851, 0.10))
    ax.tick_params(axis="y", length=0)
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.grid(axis="x", color="0.90", lw=0.4, zorder=0)
    ax.set_axisbelow(True)

    fig.legend(handles=[Line2D([], [], ls="none", marker="o", mfc="white", mec="0.3",
                               mew=1.0, ms=4.2),
                        Line2D([], [], ls="none", marker="o", color="0.3", ms=4.2)],
               labels=["base-tile split", "file-level split"],
               loc="upper center", bbox_to_anchor=(0.5, 1.0), ncol=2, fontsize=7,
               frameon=False, handlelength=1.0, handletextpad=0.4, columnspacing=1.4,
               borderaxespad=0.1, borderpad=0.0)

    fig.savefig(OUTDIR / "fig22_legacy_leakage.pdf")
    if SAVE_PNG:
        fig.savefig(OUTDIR / "fig22_legacy_leakage.png")
    print("fig22_legacy_leakage.pdf  %d x %d mm  figure  (Fig. 23)"
          % (round(W23 / MM), round(H23 / MM)))
    return fig

_f23 = build_fig23()
plt.show()


## 10 · Register, verify, zip

In [ ]:
import csv, shutil, os

CAP14 = ("Qualitative comparison on four held-out scenes. Rows 3-4 are the two "
         "architectures trained under the base-tile split, in which a scene and its "
         "mirrored copies always fall on the same side of the partition; rows 5-6 are the "
         "same two architectures trained under the file-level split, in which they do not. "
         "The rule marks a change of protocol, not a change of model. River and road "
         "fragment in rows 3-4 and are rendered almost exactly in rows 5-6, including "
         "segments that rows 3-4 break: the recovered thin structure is bought by the "
         "leakage, so the file-level panels measure recall of mirrored training tiles "
         "rather than generalisation.")

CAP23 = ("Effect of the partition protocol on reported accuracy. For each architecture the "
         "open marker is the base-tile split and the filled marker the file-level split; "
         "the arrow is the difference and the grey figure is the proportion of validation "
         "files whose mirror sits in the training partition. The two file-level runs leaked "
         "by different amounts, so they are not compared with each other. One fold and one "
         "seed, so no interval is shown.")

REGISTER = [
    dict(file="fig09_qualitative_comparison.pdf", figure="Fig. 14", section="Section 6.2",
         insertion_point="end of the subsection, replacing the existing qualitative figure",
         float_env="figure*", size="156 x 232 mm", caption=CAP14),
    dict(file="fig22_legacy_leakage.pdf", figure="Fig. 23", section="Section 6 (leakage)",
         insertion_point="beside the paragraph reporting the file-level split",
         float_env="figure", size="84 x 46 mm", caption=CAP23),
]
with open(OUTDIR / "figure_register.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(REGISTER[0]))
    w.writeheader(); w.writerows(REGISTER)

# ---- page-size check: 442.2 x 657.6 pt = 156 x 232 mm, 238.1 x 130.4 pt = 84 x 46 mm
try:
    from pypdf import PdfReader
except Exception:
    try:
        from PyPDF2 import PdfReader
    except Exception:
        PdfReader = None
if PdfReader is not None:
    for r in REGISTER:
        b = PdfReader(str(OUTDIR / r["file"])).pages[0].mediabox
        print("%-34s %6.1f x %6.1f pt  =  %.1f x %.1f mm"
              % (r["file"], float(b.width), float(b.height),
                 float(b.width) / 72 * 25.4, float(b.height) / 72 * 25.4))
else:
    print("pypdf not installed - sizes were printed by the builders above")

dest = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
zp = shutil.make_archive(os.path.join(dest, "geonusaf_two_figures"), "zip", str(OUTDIR))
print("\nzipped ->", zp)
try:
    from IPython.display import FileLink, display
    display(FileLink(os.path.relpath(zp)))
except Exception:
    pass

print("\nBefore submitting:  pdffonts figs/fig09_qualitative_comparison.pdf")
print("   -> the serif family must be embedded and no font may be Type 3.")
print("Insert with \\includegraphics[width=\\linewidth] and no scale factor.")


## 11 · Two things to settle before these go in the manuscript

**1. The numbers in Fig. 23 and the checkpoints in Fig. 14 must come from the same runs.**
`0.4657` and `0.5582` are the **block** fold-1 results; the repositories in the control panel
are the **random** fold-1 runs. Both are base-tile splits, so either is defensible, but the
figure pair has to pick one. Cell 6 prints each checkpoint's stored mIoU as it loads —
compare it with `MIOU` and change whichever is wrong. Switching the two base repos to
`…-block-fold1` is the one-line alternative.

**2. Confirm the B0 legacy fold count from its model card.** `FILE_SPLIT` now assumes
`KFold(3)` for both file-level runs, which reproduces the 91.9 % leakage you reported for B0
and which the TC card confirms for the proposed model. The B0 legacy notebook's own config
default was `KFold(5)`, which reproduces 98.8 % instead — a different validation partition
entirely, and one that would make rows 5 and 6 not directly comparable. Cell 5 measures it,
so a wrong setting shows up as a shouted mismatch rather than a quiet error.

**On the file-level mIoU for the proposed model:** `0.8393` is the pooled 6-real-class value,
which is the same estimator as the B0 file-level `0.7806`. The card's `0.8156` is the
batch-averaged estimator and `0.9674` is the CSA-core region — neither is comparable with the
B0 figure, so neither belongs on this axis.

And the standing caution about the scenes: read the audit table in cell 5 before accepting
them. A scene marked `TRAIN` in a base-tile column was trained on by that model, so rows 3–4
would not be held-out predictions for it.